In [1]:
import logging
from dataclasses import asdict
import sys
from pathlib import Path

from dotenv import load_dotenv
import pandas as pd
from datasets import load_dataset
from IPython.display import display

logging.basicConfig(level=logging.INFO)

In [2]:
from datasets import load_dataset

dataset = load_dataset(
    "SWE-Gym/OpenHands-Sampled-Trajectories",
    split="train.raw",
)

df = dataset.to_pandas()
# df = dataset.to_pandas().merge(
#     load_dataset("SWE-Gym/SWE-Gym", split="train").to_pandas(),
#     on="instance_id",
#     how="left",
# )

print(f"Trajectories: {len(dataset):,} | Combined rows: {len(df):,}")
print(df.columns.tolist())

Trajectories: 6,055 | Combined rows: 6,055
['instance_id', 'run_id', 'resolved', 'messages', 'tools', 'test_result']


## Finish calls and benchmark results across trajectories

This compares recorded assistant `finish` calls with `test_result.report.resolved`. Percentages use all loaded attempts as the denominator. A disagreement is a starting point for investigation; it does not establish that the benchmark label is wrong.


In [3]:
from collections.abc import Mapping
from importlib import reload

import numpy as np
import src.message_parse as message_parse

reload(message_parse)

print("[STAGE] Build one comparison row per trajectory")
print(f"  input trajectories: {len(dataset):,} | dataframe rows: {len(df):,}")
assert len(dataset) == len(df) == 6055, "Expected the complete 6,055-attempt split."

reports = [
    result.get("report") if isinstance(result, Mapping) else None
    for result in df["test_result"]
]
report_resolved = [
    report.get("resolved") if isinstance(report, Mapping) else None
    for report in reports
]
trace_comparison_df = df[["instance_id", "run_id"]].reset_index(drop=True).copy()
trace_comparison_df.insert(0, "trace_row", range(len(trace_comparison_df)))
trace_comparison_df["finish_called"] = pd.array(
    [message_parse.detect_finish_call(messages) for messages in df["messages"]],
    dtype="boolean",
)
trace_comparison_df["report_resolved"] = pd.array(
    [value if isinstance(value, (bool, np.bool_)) else None for value in report_resolved],
    dtype="boolean",
)
for flag in ("empty_generation", "error_eval", "failed_apply_patch", "test_timeout"):
    trace_comparison_df[f"eval_{flag}"] = [
        report.get(flag) if isinstance(report, Mapping) else None for report in reports
    ]
print(f"✓ comparison rows: {len(trace_comparison_df):,}")


[STAGE] Build one comparison row per trajectory
  input trajectories: 6,055 | dataframe rows: 6,055
✓ comparison rows: 6,055


In [ ]:
print("[STAGE] Compare finish calls with benchmark labels")
total_attempts = len(trace_comparison_df)
comparison_counts = pd.crosstab(
    trace_comparison_df["finish_called"],
    trace_comparison_df["report_resolved"],
).reindex(index=[False, True], columns=[False, True], fill_value=0)
comparison_counts.index.name = "finish_called"
comparison_counts.columns.name = "report_resolved"
comparison_table = comparison_counts.astype(object)
for finish in (False, True):
    for resolved in (False, True):
        count = int(comparison_counts.loc[finish, resolved])
        comparison_table.loc[finish, resolved] = f"{count:,} ({100 * count / total_attempts:.2f}%)"
print("✓ Cells show count (percentage of all attempts)")
display(comparison_table)

[STAGE] Compare finish calls with benchmark labels
✓ Cells show count (percentage of all attempts)


report_resolved,False,True
finish_called,,
False,"3,262 (53.87%)",37 (0.61%)
True,"2,302 (38.02%)",454 (7.50%)


In [5]:
print("[STAGE] Account for missing information")
missing_label = trace_comparison_df["report_resolved"].isna()
unknown_finish = trace_comparison_df["finish_called"].isna()
missing_labels_df = pd.DataFrame({
    "finish_called": ["False", "True", "Unknown"],
    "attempts": [
        int((missing_label & trace_comparison_df["finish_called"].eq(False).fillna(False)).sum()),
        int((missing_label & trace_comparison_df["finish_called"].eq(True).fillna(False)).sum()),
        int((missing_label & unknown_finish).sum()),
    ],
})
missing_labels_df["percent_total"] = missing_labels_df["attempts"] / total_attempts * 100
missing_labels_df["percent_total"] = missing_labels_df["percent_total"].round(2)
display(missing_labels_df)
unknown_finish_known_label = int((unknown_finish & ~missing_label).sum())
accounted = int(comparison_counts.to_numpy().sum()) + int(missing_label.sum()) + unknown_finish_known_label
print(f"  missing evaluation labels: {int(missing_label.sum()):,}")
print(f"  unknown finish status with a known label: {unknown_finish_known_label:,}")
print(f"✓ accounted for {accounted:,} of {total_attempts:,} attempts")
assert accounted == total_attempts


[STAGE] Account for missing information


,finish_called,attempts,percent_total
0,False,0,0.0
1,True,0,0.0
2,Unknown,0,0.0


  missing evaluation labels: 0
  unknown finish status with a known label: 0
✓ accounted for 6,055 of 6,055 attempts


In [6]:
print("[STAGE] Select examples for manual investigation")
example_columns = [
    "trace_row", "instance_id", "run_id", "finish_called", "report_resolved",
    "eval_empty_generation", "eval_error_eval", "eval_failed_apply_patch", "eval_test_timeout",
]
finish_unresolved = trace_comparison_df.loc[
    trace_comparison_df["finish_called"].eq(True).fillna(False)
    & trace_comparison_df["report_resolved"].eq(False).fillna(False), example_columns
].head(3)
no_finish_resolved = trace_comparison_df.loc[
    trace_comparison_df["finish_called"].eq(False).fillna(False)
    & trace_comparison_df["report_resolved"].eq(True).fillna(False), example_columns
].head(3)
print(f"✓ finish=True, resolved=False: {len(finish_unresolved)} examples")
display(finish_unresolved)
print(f"✓ finish=False, resolved=True: {len(no_finish_resolved)} examples")
display(no_finish_resolved)


[STAGE] Select examples for manual investigation
✓ finish=True, resolved=False: 3 examples


,trace_row,instance_id,run_id,finish_called,report_resolved,eval_empty_generation,eval_error_eval,eval_failed_apply_patch,eval_test_timeout
10,10,python__mypy-11420,gpt-4o-2024-08-06_maxiter_30_N_v2.1-no-hint-tr...,True,False,True,False,False,False
13,13,getmoto__moto-6178,gpt-4o-2024-08-06_maxiter_30_N_v2.1-no-hint-tr...,True,False,True,False,False,False
48,48,iterative__dvc-5004,gpt-4o-2024-08-06_maxiter_30_N_v2.1-no-hint-tr...,True,False,True,False,False,False


✓ finish=False, resolved=True: 3 examples


,trace_row,instance_id,run_id,finish_called,report_resolved,eval_empty_generation,eval_error_eval,eval_failed_apply_patch,eval_test_timeout
128,128,getmoto__moto-6913,gpt-4o-2024-08-06_maxiter_30_N_v2.1-no-hint-tr...,False,True,False,False,False,False
202,202,iterative__dvc-5839,gpt-4o-2024-08-06_maxiter_30_N_v2.1-no-hint-tr...,False,True,False,False,False,False
388,388,conan-io__conan-13403,gpt-4o-2024-08-06_maxiter_30_N_v2.1-no-hint-tr...,False,True,False,False,False,False


### 1.b Inspect messages in order

Each row is a recorded message. `tool_call_id` links a tool response to its call when present.

In [30]:
sample_index = 150
sample = df.iloc[sample_index]
print(f"[STAGE] Inspect record {sample_index}")
print("✓ Fields:", df.columns)
display(pd.Series({key: sample.get(key) for key in ["instance_id", "run_id", "resolved", "test_result"]}, name="value").to_frame())
messages = sample.get("messages")
tools = sample.get("tools")
print("Messages:", len(messages) if messages is not None else 0)
print("Available tools:", [tool.get("function", {}).get("name") for tool in (tools if tools is not None else [])])
sample['test_result']['apply_patch_output']

[STAGE] Inspect record 150
✓ Fields: Index(['instance_id', 'run_id', 'resolved', 'messages', 'tools',
       'test_result'],
      dtype='object')


,value
instance_id,Project-MONAI__MONAI-3547
run_id,gpt-4o-2024-08-06_maxiter_30_N_v2.1-no-hint-tr...
resolved,False
test_result,{'apply_patch_output': 'Checking patch monai/u...


Messages: 21
Available tools: ['execute_bash', 'finish', 'str_replace_editor']


'Checking patch monai/utils/misc.py...\r\nChecking patch reproduce_error.py...\r\nApplied patch monai/utils/misc.py cleanly.\r\nApplied patch reproduce_error.py cleanly.\r\nAPPLY_PATCH_PASS'

In [8]:
display(sample.drop("messages").to_frame(name="value"))

,value
instance_id,python__mypy-10658
run_id,gpt-4o-2024-08-06_maxiter_30_N_v2.1-no-hint-tr...
resolved,False
tools,[{'function': {'description': 'Execute a bash ...
test_result,{'apply_patch_output': 'Checking patch reprodu...


'Checking patch reproduce_error.py...\r\nApplied patch reproduce_error.py cleanly.\r\nAPPLY_PATCH_PASS'

### Creating the DF for understanding a message

In [10]:
from src.message_parse import (
    attach_tool_responses,
    enrich_tool_interactions,
    extract_tool_call_df,
    prepare_message_df,
)


### Step 1: Prepare messages
One row for each original message.

In [12]:
print("[STAGE] Prepare message rows")
prepared_message_df = prepare_message_df(sample, messages)
print(f"✓ message rows: {len(prepared_message_df)}")
# display(prepared_message_df)

[STAGE] Prepare message rows
✓ message rows: 61


### Step 2: Extract tool calls
One row for each assistant tool call.

In [13]:
print("[STAGE] Extract assistant tool calls")
tool_call_df = extract_tool_call_df(prepared_message_df)
print(f"✓ tool call rows: {len(tool_call_df)}")
# display(tool_call_df)

[STAGE] Extract assistant tool calls
✓ tool call rows: 30


### Step 3: Attach tool responses
Match calls to responses by call ID and retain other messages.

In [15]:
print("[STAGE] Attach tool responses")
print(f"  input messages: {len(prepared_message_df)} | calls: {len(tool_call_df)}")
tool_interactions_df = attach_tool_responses(prepared_message_df, tool_call_df)
print(f"✓ interaction rows: {len(tool_interactions_df)}")
# display(tool_interactions_df)

[STAGE] Attach tool responses
  input messages: 61 | calls: 30
✓ interaction rows: 32


### Step 4: Expand arguments and tool definitions
Build the final DataFrame with argument details and matching tool definitions.

In [16]:
print("[STAGE] Expand arguments and attach tool definitions")
print(f"  input interaction rows: {len(tool_interactions_df)}")
final_message_df = enrich_tool_interactions(tool_interactions_df, tools)
print(f"✓ final rows: {len(final_message_df)}")
with pd.option_context("display.max_columns", None):
    display(final_message_df)

INFO:src.message_parse:enrich_tool_interactions | input_rows=32
INFO:src.message_parse:enrich_tool_interactions | output_rows=32


[STAGE] Expand arguments and attach tool definitions
  input interaction rows: 32
✓ final rows: 32


,content,function_call,name,role,tool_call_id,tool_calls,instance_id,run_id,message_index,row_type,message,tool_call,call_id,call_type,call_name,call_arguments,response_message_index,response_message,response_matches,matched,response_content,response_function_call,response_name,response_role,response_tool_call_id,response_tool_calls,arg_command,arg_path,arg_view_range,arg_file_text,argument_parse_error,tool_definition_type,tool_definition_function.description,tool_definition_function.name,tool_definition_function.parameters.properties.command.description,tool_definition_function.parameters.properties.command.enum,tool_definition_function.parameters.properties.command.type,tool_definition_function.parameters.properties.file_text,tool_definition_function.parameters.properties.insert_line,tool_definition_function.parameters.properties.new_str,tool_definition_function.parameters.properties.old_str,tool_definition_function.parameters.properties.path,tool_definition_function.parameters.properties.view_range,tool_definition_function.parameters.required,tool_definition_function.parameters.type,tool_definition_function.parameters,tool_definition_function.parameters.properties.file_text.description,tool_definition_function.parameters.properties.file_text.type,tool_definition_function.parameters.properties.insert_line.description,tool_definition_function.parameters.properties.insert_line.type,tool_definition_function.parameters.properties.new_str.description,tool_definition_function.parameters.properties.new_str.type,tool_definition_function.parameters.properties.old_str.description,tool_definition_function.parameters.properties.old_str.type,tool_definition_function.parameters.properties.path.description,tool_definition_function.parameters.properties.path.type,tool_definition_function.parameters.properties.view_range.description,tool_definition_function.parameters.properties.view_range.items.type,tool_definition_function.parameters.properties.view_range.type
0,You are a helpful assistant that can interact ...,None,None,system,None,None,python__mypy-10658,gpt-4o-2024-08-06_maxiter_30_N_v2.1-no-hint-tr...,0,message,{'content': 'You are a helpful assistant that ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,<uploaded_files>\n/workspace/python__mypy__0.9...,None,None,user,None,None,python__mypy-10658,gpt-4o-2024-08-06_maxiter_30_N_v2.1-no-hint-tr...,1,message,{'content': '<uploaded_files> /workspace/pytho...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,,None,None,assistant,None,"[{'function': {'arguments': '{""command"":""view""...",python__mypy-10658,gpt-4o-2024-08-06_maxiter_30_N_v2.1-no-hint-tr...,2,tool_interaction,"{'content': '', 'function_call': None, 'name':...","{'function': {'arguments': '{""command"":""view"",...",call_TMLO3CTA44gY963iU3kAtwQ6,function,str_replace_editor,"{""command"":""view"",""path"":""/workspace/python__m...",3.0,{'content': 'OBSERVATION: Here's the files and...,"[{'message_index': 3, 'message': {'content': '...",True,OBSERVATION:\nHere's the files and directories...,NaN,str_replace_editor,tool,call_TMLO3CTA44gY963iU3kAtwQ6,NaN,view,/workspace/python__mypy__0.910,NaN,NaN,None,function,"Custom editing tool for viewing, creating and ...",str_replace_editor,The commands to run. Allowed options are: `vie...,"[view, create, str_replace, insert, undo_edit]",string,NaN,NaN,NaN,NaN,NaN,NaN,"[command, path]",object,NaN,"Required parameter of `create` command, with t...",string,Required parameter of `insert` command. The `n...,integer,Optional parameter of `str_replace` command co...,string,Required parameter of `str_replace` command co...,string,"Absolute path to file or directory, e.g. `/rep...",string,Optional parameter o

In [18]:
sample['test_result']

{'apply_patch_output': 'Checking patch reproduce_error.py...\r\nApplied patch reproduce_error.py cleanly.\r\nAPPLY_PATCH_PASS',
 'git_patch': 'diff --git a/reproduce_error.py b/reproduce_error.py\r\nnew file mode 100644\r\nindex 000000000..faffa90e3\r\n--- /dev/null\r\n+++ b/reproduce_error.py\r\n@@ -0,0 +1,13 @@\r\n+from typing import TypeVar, Type\r\n+\r\n+class A:\r\n+    a: int = 1\r\n+\r\n+T = TypeVar("T", bound=A)\r\n+\r\n+def foo(t: Type[T], a: A) -> None:\r\n+    if type(a) is t:\r\n+        reveal_type(a)  # <nothing>, with 0.8xx it was A\r\n+        a.a  # error: <nothing> has no attribute "a"\r\n+\r\n+foo(A, A())\r\n\\ No newline at end of file',
 'report': {'empty_generation': False,
  'error_eval': False,
  'failed_apply_patch': False,
  'resolved': False,
  'test_timeout': False},
 'test_output': '+ source /opt/miniconda3/bin/activate\r\n++ _CONDA_ROOT=/opt/miniconda3\r\n++ . /opt/miniconda3/etc/profile.d/conda.sh\r\n+++ export CONDA_EXE=/opt/miniconda3/bin/conda\r\n+++ C

In [ ]:
# sample['test_result']['apply_patch_output'], sample['test_result']['git_patch']
# sample['test_result']['report']
# sample['test_result']['test_output']

### Additional exploration (optional)
The cells below are earlier exploratory work.

In [ ]:
# PYTHONPATH=src ./financial_env/bin/python scripts/workbench/create_readonly_role.py --apply

In [ ]:
# 1. Recommended learning order
# 1. Stage 2 — Trajectory quality and failure analysis
   
#    Implement quality metrics, identify wasted actions, distinguish recovered errors from unresolved failures, and construct a filtered dataset.
   
#    Deliverable: trajectory-quality evaluator
# 2. Stage 3A — Deterministic verifiers
   
#    Build independent checks for tool execution, test outcomes, and task completion. Explicitly track when evidence is missing.
   
#    Deliverable: structured verification results
# 3. Stage 3B — Reward design and GRPO
   
#    Turn reliable verification results into rewards. Test reward hacking, calculate group-relative advantages, and compare attempts on the same issue.
   
#    Deliverable: reward evaluator + NumPy/PyTorch implementation
# 4. Stage 4 — Reliable execution
   
#    Implement retries, timeouts, circuit breakers, and stateful tool execution. Only use isolated environments for replaying commands.
   
#    Deliverable: small tool execution controller

### Final DataFrame